In [ ]:
import polars as pl

In [ ]:
exclude_amc = [
    "DSPAMC",
    "EDELAMC",
    "GROWWAMC",
    "HDFCAMC",
    "MIRAEAMC",
    "SBIAMC",
    "UTIAMC",
    "ZERODHAAMC",
]

exclude_conditions = [
    pl.col("name").str.contains(pattern)
    for pattern in ["%", "GOI TBILL", "ETF", *exclude_amc]
] + [pl.col("symbol").str.ends_with("INAV")]

df = (
    pl.scan_parquet("kite_instruments.parquet")
    .select("instrument_token", "symbol", "name", "tick_size", "segment")
    .with_columns(
        pl.col("symbol")
        .str.split("-")
        .list.get(1, null_on_oob=True)
        .fill_null("EQ")
        .alias("suffix")
    )
    .filter(
        (pl.col("segment") == "NSE")
        & (pl.col("name").str.len_chars() > 0)
        & (pl.col("suffix") == "EQ")
        & ~pl.any_horizontal(exclude_conditions)
    )
    .drop("segment", "suffix")
    .collect()
)

df